# Notebook 02: AugMix Robustness Baseline

AugMix creates multiple stochastic augmentation chains, mixes them, and adds a Jensen-Shannon consistency penalty. This is a strong published baseline for common corruption robustness.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import torch
import matplotlib.pyplot as plt

from robust_cifar.data import make_cifar10_loaders, CIFAR10_CLASSES
from robust_cifar.models import build_resnet18_cifar, count_parameters
from robust_cifar.train import get_device, seed_everything

seed_everything(42)
device = get_device()
device

device(type='cuda')

## AugMix Loader

Each batch returns a clean image plus two AugMix views used for JSD consistency training.

In [2]:
train_loader, test_loader = make_cifar10_loaders(
    data_dir=PROJECT_ROOT / "data",
    batch_size=64,
    num_workers=2,
    mode="augmix",
    download=True,
)
batch = next(iter(train_loader))
[x.shape if hasattr(x, "shape") else len(x) for x in batch]

Files already downloaded and verified
Files already downloaded and verified


[torch.Size([64, 3, 32, 32]),
 torch.Size([64, 3, 32, 32]),
 torch.Size([64, 3, 32, 32]),
 torch.Size([64])]

## Train AugMix Model

In [ ]:
from robust_cifar.train import train_augmix

model = build_resnet18_cifar()
EPOCHS = 100
history = train_augmix(
    model,
    train_loader,
    test_loader,
    epochs=EPOCHS,
    lr=0.1,
    jsd_weight=12.0,
    device=device,
    output_dir=PROJECT_ROOT / "checkpoints",
    run_name="resnet18_augmix",
)
metrics_df = (
    pd.DataFrame(history)[["epoch", "train_loss", "train_accuracy", "val_loss", "val_accuracy"]]
    .rename(columns={"val_loss": "test_loss", "val_accuracy": "test_accuracy"})
)

for row in metrics_df.itertuples(index=False):
    print(
        f"Epoch {int(row.epoch):03d} | "
        f"Train Loss: {row.train_loss:.4f} | Train Acc: {row.train_accuracy:.4f} | "
        f"Test Loss: {row.test_loss:.4f} | Test Acc: {row.test_accuracy:.4f}"
    )

metrics_df

resnet18_augmix epoch 1/100:   0%|          | 0/782 [00:11<?, ?it/s]

resnet18_augmix epoch 2/100:   0%|          | 0/782 [00:14<?, ?it/s]

resnet18_augmix epoch 3/100:   0%|          | 0/782 [00:13<?, ?it/s]

resnet18_augmix epoch 4/100:   0%|          | 0/782 [00:13<?, ?it/s]

## Training and Testing Accuracy Comparison

The table above reports epoch, loss, and accuracy for both training and testing data. The graph below compares training accuracy against testing accuracy across epochs.

In [ ]:
from robust_cifar.visualize import plot_training_history

history_df = pd.DataFrame(history)
plot_training_history(
    history_df,
    title="AugMix ResNet-18",
    output_path=PROJECT_ROOT / "reports" / "figures" / "augmix_training.png",
)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(metrics_df["epoch"], metrics_df["train_accuracy"], label="Training accuracy", linewidth=2)
ax.plot(metrics_df["epoch"], metrics_df["test_accuracy"], label="Testing accuracy", linewidth=2)
ax.set_title("AugMix ResNet-18: Training vs Testing Accuracy")
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports" / "figures" / "augmix_train_test_accuracy.png", dpi=180, bbox_inches="tight")
plt.show()